# A. Острова рекомендаций — разбор

Цель — получить ответы на A1–A5 из файлов items.csv и also_viewed.csv. Положите оба файла рядом с ноутбуком и запустите все ячейки. При запуске из корня репозитория код также проверяет папку task_vseros_0.

**Важно:** в A4 учитывается только столбец item_to. В A5 каждая пара товаров задаёт **неориентированную** связь: идти по ней можно в обе стороны.

Пока исходных CSV в папке нет, поэтому ниже приведён готовый код и небольшой проверочный пример. Конкретные ответы появятся после добавления таблиц.

## 0. Загрузка данных

Проверяем уникальность item_id и то, что все идентификаторы из таблицы связей есть среди товаров. Иначе категория некоторых вершин графа будет неизвестна.

In [ ]:
from pathlib import Path
import pandas as pd

candidates = (
    Path.cwd(),
    Path.cwd() / "task_vseros_0",
    Path.cwd() / "task_vseros",
)
data_dir = next(
    (
        folder for folder in candidates
        if (folder / "items.csv").is_file()
        and (folder / "also_viewed.csv").is_file()
    ),
    None,
)

items = also_viewed = None
if data_dir is None:
    print("CSV пока не найдены. Положите items.csv и also_viewed.csv рядом с ноутбуком.")
else:
    items = pd.read_csv(data_dir / "items.csv")
    also_viewed = pd.read_csv(data_dir / "also_viewed.csv")

    required_items = {"item_id", "category", "price", "rating", "brand", "in_stock"}
    required_links = {"item_from", "item_to"}
    if not required_items.issubset(items.columns):
        raise ValueError(f"В items.csv нет столбцов: {required_items - set(items.columns)}")
    if not required_links.issubset(also_viewed.columns):
        raise ValueError(f"В also_viewed.csv нет столбцов: {required_links - set(also_viewed.columns)}")
    if not items["item_id"].is_unique:
        raise ValueError("item_id должен быть уникальным")

    known_ids = set(items["item_id"])
    link_ids = set(also_viewed["item_from"]) | set(also_viewed["item_to"])
    if not link_ids.issubset(known_ids):
        raise ValueError(f"В связях есть неизвестные item_id: {sorted(link_ids - known_ids)[:10]}")
    print(f"Товаров: {len(items)}, строк со связями: {len(also_viewed)}")
    print("Категории:", sorted(items["category"].unique()))

## A1. Телефоны с высоким рейтингом в наличии

Нужны товары, для которых **одновременно** верны три условия. Знак & соединяет булевы условия логическим «и».

In [ ]:
def answer_a1(items):
    suitable = (
        items["category"].eq("phones")
        & items["rating"].ge(4.5)
        & items["in_stock"].eq(1)
    )
    return int(suitable.sum())

## A2. Бренд ноутбуков с наибольшей средней ценой

Оставляем только laptops, группируем по brand и считаем **среднюю цену**, а не сумму цен или цену самого дорогого ноутбука. Если лидеров несколько, выбираем первый бренд по алфавиту: условие допускает любой.

In [ ]:
def laptop_mean_prices(items):
    laptops = items.loc[items["category"].eq("laptops")]
    return laptops.groupby("brand")["price"].mean().sort_index()

def answer_a2(items):
    means = laptop_mean_prices(items)
    if means.empty:
        raise ValueError("В items.csv нет товаров категории laptops")
    winners = means.index[means.eq(means.max())]
    return sorted(winners)[0]

## A3. Сегменты

Premium: rating ≥ 4.5 и price ≥ 50000. Standard: rating ≥ 4.0 и price < 50000. Все остальные — budget. После разметки считаем только premium, имеющиеся в наличии.

In [ ]:
def add_segments(items):
    result = items.copy()
    segment = pd.Series("budget", index=items.index, name="segment")
    standard = items["rating"].ge(4.0) & items["price"].lt(50_000)
    premium = items["rating"].ge(4.5) & items["price"].ge(50_000)
    segment.loc[standard] = "standard"
    segment.loc[premium] = "premium"
    result["segment"] = segment
    return result

def answer_a3(items):
    segmented = add_segments(items)
    return int(
        (segmented["segment"].eq("premium") & segmented["in_stock"].eq(1)).sum()
    )

## A4. Разные товары из item_to по категориям

Повторные появления одного item_to нельзя считать несколько раз. Превращаем значения item_to в множество и отмечаем товары, входящие в него. Группируем **все** товары из items.csv: так категории с нулём тоже окажутся в таблице. В этом пункте не разворачиваем связи в обратную сторону.

In [ ]:
def answer_a4(items, also_viewed):
    target_ids = set(also_viewed["item_to"])
    marked = items[["category", "item_id"]].copy()
    marked["in_item_to"] = marked["item_id"].isin(target_ids).astype(int)
    table = (
        marked.groupby("category", sort=True, as_index=False)["in_item_to"]
        .sum()
        .rename(columns={"in_item_to": "cnt"})
    )
    table["cnt"] = table["cnt"].astype(int)
    return table[["category", "cnt"]]

## A5. Острова рекомендаций

Остров — это **компонента связности** неориентированного графа. Товары являются вершинами, каждая пара из also_viewed.csv — ребром в обе стороны. Обходим граф поиском в глубину. Для каждой компоненты собираем категории и прибавляем 1, если среди них есть и phones, и accessories.

Изолированный товар тоже образует компоненту, поэтому словарь соседей создаём сразу для **всех** товаров. Время работы — O(V + E), где V — число товаров, E — число связей.

In [ ]:
def answer_a5(items, also_viewed):
    category_by_id = dict(zip(items["item_id"], items["category"]))
    neighbors = {item_id: set() for item_id in category_by_id}

    for source, target in also_viewed[["item_from", "item_to"]].itertuples(index=False, name=None):
        neighbors[source].add(target)
        neighbors[target].add(source)  # Связь работает в обе стороны.

    visited = set()
    suitable_islands = 0
    for start in neighbors:
        if start in visited:
            continue
        stack = [start]
        categories = set()
        while stack:
            current = stack.pop()
            if current in visited:
                continue
            visited.add(current)
            categories.add(category_by_id[current])
            stack.extend(neighbors[current] - visited)
        if {"phones", "accessories"}.issubset(categories):
            suitable_islands += 1
    return suitable_islands

### Проверочный пример для A4 и A5

Пусть связи образуют острова 1—2—3, 4—5 и одиночный товар 6. В первых двух есть телефон и аксессуар, поэтому A5 = 2. Товар 1 при этом не встречается в item_to, хотя связан с товаром 2: для A4 это важно.

In [ ]:
example_items = pd.DataFrame({
    "item_id": [1, 2, 3, 4, 5, 6],
    "category": ["phones", "accessories", "books", "phones", "accessories", "toys"],
})
example_links = pd.DataFrame({
    "item_from": [1, 2, 4, 5],
    "item_to": [2, 3, 5, 4],
})
print("A4 на примере:")
print(answer_a4(example_items, example_links).to_string(index=False))
print("A5 на примере:", answer_a5(example_items, example_links))

## Ответы для реальных данных

Следующая ячейка выдаёт A1, A2, A3, A5 и создаёт файл answer4.csv **рядом с исходными таблицами**. Файл A4 содержит только столбцы category,cnt, по одной строке на категорию в алфавитном порядке.

In [ ]:
if items is None:
    print("Для численных ответов добавьте items.csv и also_viewed.csv, затем запустите ноутбук заново.")
else:
    print("A1:", answer_a1(items))
    means = laptop_mean_prices(items)
    print("Средние цены ноутбуков по брендам:")
    print(means.sort_values(ascending=False).to_string())
    print("A2:", answer_a2(items))
    print("A3:", answer_a3(items))

    answer4 = answer_a4(items, also_viewed)
    answer4_path = data_dir / "answer4.csv"
    answer4.to_csv(answer4_path, index=False)
    print("A4 — сохранено в:", answer4_path)
    print(answer4.to_string(index=False))
    print("A5:", answer_a5(items, also_viewed))

### Перед сдачей

Ответы A1, A2, A3 и A5 появятся в последней ячейке. Для A4 загрузите созданный answer4.csv. В нём должна быть строка для каждой категории из items.csv, даже если cnt = 0.